# Final Project — Step 2: Silver Transformation
## Source: Bronze Delta | Target: Silver Delta

Two-stage data quality framework:
1. **Completeness check** — detect NULLs in critical fields → quarantine with reason `INCOMPLETE_NULL_IN_CRITICAL_FIELD`
2. **Validity check** — apply business rules on complete rows → quarantine with granular reason codes

Valid rows are enriched with derived columns (`trip_duration_minutes`, `pickup_hour`, `is_weekend`, `avg_speed_mph`)
then validated through a Great Expectations gate before Silver write.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("final-project-silver")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print("✓ Spark session created with Delta support")

Spark version: 3.5.0
✓ Spark session created with Delta support


In [2]:
# Paths
BASE_PATH           = "/workspace/output/final_project"
BRONZE_YELLOW_PATH  = f"{BASE_PATH}/bronze/yellow"
SILVER_YELLOW_PATH  = f"{BASE_PATH}/silver/yellow"
QUARANTINE_PATH     = f"{BASE_PATH}/quarantine/yellow"

print(f"Source    : {BRONZE_YELLOW_PATH}")
print(f"Target    : {SILVER_YELLOW_PATH}")
print(f"Quarantine: {QUARANTINE_PATH}")

Source    : /workspace/output/final_project/bronze/yellow
Target    : /workspace/output/final_project/silver/yellow
Quarantine: /workspace/output/final_project/quarantine/yellow


In [3]:
# Read Bronze
df_bronze = spark.read.format("delta").load(BRONZE_YELLOW_PATH)
print(f"Bronze record count: {df_bronze.count():,}")

Bronze record count: 2,964,624


In [4]:
# Stage 1: Completeness check
# Rows with NULLs in critical fields are separated BEFORE applying business rules.

from pyspark.sql.functions import col

CRITICAL_COLS = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance",
    "fare_amount", "total_amount"
]

null_predicate = None
for c in CRITICAL_COLS:
    expr = col(c).isNull()
    null_predicate = expr if null_predicate is None else (null_predicate | expr)

df_incomplete = df_bronze.filter(null_predicate)
df_complete   = df_bronze.filter(~null_predicate)

print(f"Complete   : {df_complete.count():,}")
print(f"Incomplete : {df_incomplete.count():,}")

# Verify no rows lost — critical sanity check
assert df_complete.count() + df_incomplete.count() == df_bronze.count(), \
    "Row count mismatch after completeness split"
print("✓ No rows lost in completeness split")

Complete   : 2,824,462
Incomplete : 140,162
✓ No rows lost in completeness split


In [5]:
# Stage 2: Validity check — applied only on complete rows
# Granular rejection reason codes allow root-cause analysis of data quality failures

from pyspark.sql.functions import when, lit

valid_predicate = (
    (col("passenger_count") > 0)
    & (col("trip_distance") > 0)
    & (col("trip_distance") < 500)
    & (col("fare_amount") > 0)
    & (col("total_amount") > 0)
    & (col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
)

df_valid   = df_complete.filter(valid_predicate)
df_invalid = df_complete.filter(~valid_predicate)

# Tag invalid rows with granular reason codes
df_invalid_tagged = df_invalid.withColumn(
    "quarantine_reason",
    when(col("passenger_count") <= 0,                                          lit("INVALID_PASSENGER_COUNT"))
    .when(col("trip_distance") <= 0,                                           lit("INVALID_DISTANCE_ZERO_OR_NEG"))
    .when(col("trip_distance") >= 500,                                         lit("INVALID_DISTANCE_TOO_LONG"))
    .when(col("fare_amount") <= 0,                                             lit("INVALID_FARE"))
    .when(col("total_amount") <= 0,                                            lit("INVALID_TOTAL_AMOUNT"))
    .when(col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime"),         lit("INVALID_TIME_ORDER"))
    .otherwise(                                                                lit("INVALID_OTHER"))
)

# Tag incomplete rows
df_incomplete_tagged = df_incomplete.withColumn(
    "quarantine_reason", lit("INCOMPLETE_NULL_IN_CRITICAL_FIELD")
)

bronze_count  = df_bronze.count()
valid_count   = df_valid.count()
invalid_count = df_invalid.count()
incomp_count  = df_incomplete.count()

print(f"Bronze total : {bronze_count:,}  (100.00%)")
print(f"Valid        : {valid_count:,}  ({valid_count/bronze_count*100:.2f}%)")
print(f"Invalid      : {invalid_count:,}   ({invalid_count/bronze_count*100:.2f}%)")
print(f"Incomplete   : {incomp_count:,}  ({incomp_count/bronze_count*100:.2f}%)")
print(f"Sum check    : {valid_count + invalid_count + incomp_count:,}  (must equal Bronze)")

Bronze total : 2,964,624  (100.00%)
Valid        : 2,723,745  (91.87%)
Invalid      : 100,717   (3.40%)
Incomplete   : 140,162  (4.73%)
Sum check    : 2,964,624  (must equal Bronze)


In [6]:
# Write unified quarantine table
# Both incomplete and invalid rows land here, partitioned by reason for efficient querying

df_quarantine = df_incomplete_tagged.unionByName(df_invalid_tagged)

(
    df_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("quarantine_reason")
    .save(QUARANTINE_PATH)
)

print(f"✓ Quarantine written: {QUARANTINE_PATH}")
print("\nQuarantine breakdown by reason:")
spark.read.format("delta").load(QUARANTINE_PATH) \
    .groupBy("quarantine_reason").count() \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

✓ Quarantine written: /workspace/output/final_project/quarantine/yellow

Quarantine breakdown by reason:
+---------------------------------+------+
|quarantine_reason                |count |
+---------------------------------+------+
|INCOMPLETE_NULL_IN_CRITICAL_FIELD|140162|
|INVALID_DISTANCE_ZERO_OR_NEG     |36763 |
|INVALID_FARE                     |32429 |
|INVALID_PASSENGER_COUNT          |31465 |
|INVALID_TIME_ORDER               |55    |
|INVALID_DISTANCE_TOO_LONG        |5     |
+---------------------------------+------+



In [7]:
# Enrich valid rows with derived columns used downstream in Gold KPIs

from pyspark.sql.functions import (
    unix_timestamp, hour, dayofweek, when, round as spark_round
)

df_silver = (
    df_valid
    .withColumn(
        "trip_duration_minutes",
        spark_round(
            (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60,
            2
        )
    )
    .withColumn("pickup_hour",        hour("tpep_pickup_datetime"))
    .withColumn("pickup_day_of_week", dayofweek("tpep_pickup_datetime"))
    .withColumn(
        "is_weekend",
        when(dayofweek("tpep_pickup_datetime").isin([1, 7]), True).otherwise(False)
    )
    .withColumn(
        "avg_speed_mph",
        spark_round(col("trip_distance") / (col("trip_duration_minutes") / 60), 2)
    )
)

# Second-pass filter: remove trips with unreasonable durations
df_silver = df_silver.filter(
    (col("trip_duration_minutes") > 0) & (col("trip_duration_minutes") < 1440)
)

print(f"Silver record count after enrichment: {df_silver.count():,}")
df_silver.select(
    "tpep_pickup_datetime", "trip_duration_minutes",
    "pickup_hour", "is_weekend", "avg_speed_mph"
).show(5, truncate=False)

Silver record count after enrichment: 2,723,734
+--------------------+---------------------+-----------+----------+-------------+
|tpep_pickup_datetime|trip_duration_minutes|pickup_hour|is_weekend|avg_speed_mph|
+--------------------+---------------------+-----------+----------+-------------+
|2024-01-01 00:57:55 |19.8                 |0          |false     |5.21         |
|2024-01-07 14:00:31 |19.18                |14         |true      |9.48         |
|2024-01-01 00:03:00 |6.6                  |0          |false     |16.36        |
|2024-01-07 14:21:45 |20.33                |14         |true      |14.2         |
|2024-01-01 00:17:06 |17.92                |0          |false     |15.74        |
+--------------------+---------------------+-----------+----------+-------------+
only showing top 5 rows



In [8]:
# Great Expectations validation gate
# Validates a 100k sample of the Silver DataFrame before writing.

import great_expectations as ge

sample_pdf = df_silver.limit(100_000).toPandas()
ge_df      = ge.from_pandas(sample_pdf)

results = []
results.append(ge_df.expect_column_values_to_not_be_null("tpep_pickup_datetime"))
results.append(ge_df.expect_column_values_to_not_be_null("tpep_dropoff_datetime"))
results.append(ge_df.expect_column_values_to_be_between("passenger_count",        min_value=1,  max_value=9))
results.append(ge_df.expect_column_values_to_be_between("trip_distance",          min_value=0,  max_value=500))
results.append(ge_df.expect_column_values_to_be_between("fare_amount",            min_value=0,  max_value=10000))
results.append(ge_df.expect_column_values_to_be_between("trip_duration_minutes",  min_value=0,  max_value=1440))
results.append(ge_df.expect_column_values_to_be_in_set("is_weekend",             [True, False]))

all_passed = all(r["success"] for r in results)
failed     = [r for r in results if not r["success"]]

print(f"Expectations run    : {len(results)}")
print(f"Expectations passed : {len(results) - len(failed)}")
print(f"Expectations failed : {len(failed)}")

if not all_passed:
    print("\n Failed expectations:")
    for f in failed:
        print(f"  - {f['expectation_config']['expectation_type']}: {f['result']}")
    raise ValueError("Data quality gate FAILED — Silver write aborted")

print("\n✓ All expectations passed — proceeding to Silver write")

Expectations run    : 7
Expectations passed : 7
Expectations failed : 0

✓ All expectations passed — proceeding to Silver write


In [9]:
# Write Silver, partitioned by pickup_hour
# Gold queries frequently filter/group by hour 

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_hour")
    .save(SILVER_YELLOW_PATH)
)

print(f"✓ Silver write complete: {SILVER_YELLOW_PATH}")

✓ Silver write complete: /workspace/output/final_project/silver/yellow


In [10]:
# Final verification
df_silver_check = spark.read.format("delta").load(SILVER_YELLOW_PATH)

print(f"Silver count  : {df_silver_check.count():,}")
print(f"Silver columns: {len(df_silver_check.columns)}")

print("\nDelta history:")
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_YELLOW_PATH}`").show(truncate=False)

Silver count  : 2,723,734
Silver columns: 28

Delta history:
+-------+-----------------------+---------------+-------------+---------+-------------------------------------------------------------------------+----+------------------+--------------------+-----------+-----------------+-------------+----------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp              |userId         |userName     |operation|operationParameters                                                      |job |notebook          |clusterId           |readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                      |userMetadata|engineInfo                                |
+-------+-----------------------+---------------+-------------+---------+-------------------------------------------------------------------------+----+------------------+--------------------+-----------+-------